# Jesuit Entry Analysis (`jesuita-entrada`)

This notebook analyzes the places where Jesuit missionaries entered the Society of Jesus (novitiate locations), exploring recruitment patterns, geographic origins, and career trajectories.

**Analysis Workflow:**

1.  **Global Trends**:
    *   **Frequency Analysis**: Counts the number of entries per location and time period.
    *   **Geographic Distribution**: Links entry places and birthplaces to Wikidata IDs for mapping.
    *   **Temporal Analysis**: Visualizes recruitment trends by decade (histogram of entries per 10-year period).

2.  **Specific Place Focus (e.g., Coimbra)**:
    *   Filters missionaries who entered at a specific college (variable `place_of_entry`).
    *   **Demographics**: Calculates "Age at Entry", "Age at Departure" (to missions), and "Years of Service".
    *   **Origins**: Cross-references entry place with birthplace to identify local recruits vs. those who traveled to enter.
    *   **Career**: tracks roles (`jesuita-cargo`) and tasks (`jesuita-tarefa`) performed by these specific cohorts.

3.  **Data Quality & Enrichment**:
    *   Identifies records missing Wikidata links for places.
    *   Isolates entries with "Unknown" places of entry (`?`) and suggests candidates based on birthplace proximity.

**Key Outputs (`../inferences/`):**
*   `jesuitas-entrada-full.xlsx`: Complete dataset of entry records with dates and locations.
*   `jesuita-entrada-{place}.xlsx`: Detailed list for a specific location (e.g., Coimbra).
*   `jesuita-entrada-{place}-cargos_tarefas.xlsx`: Roles held by missionaries from that college.
*   `{place}/markdown/*.md`: Individual markdown biography stubs for missionaries from the selected place.

In [ ]:
from timelink.notebooks import TimelinkNotebook

from timelink import version as timelink_version

tlnb = TimelinkNotebook()
print(f"Timelink version: {timelink_version}")


The attribute `jesuita-entrada` records the jesuit college/noviciate where the future missionary enrolled in the Society of Jesus.

Note that a few entries have more than one value for `jesuita-entrada` because the person dropped out after the first enrollement and later enrolled again.

## Number of entries per location, dates

In [ ]:
import pandas as pd
from timelink.pandas import attribute_values
from timelink.kleio.utilities import format_timelink_date, convert_timelink_date

# set max rows to 500
pd.set_option('display.max_rows', 500)
show_only = 30

df_totals = attribute_values('jesuita-entrada',db=tlnb.db)
# df_totals.info()
df_totals['data_inicial'] = df_totals['date_min'].apply(format_timelink_date)
df_totals['data_final'] = df_totals['date_max'].apply(format_timelink_date)
# Extract the first 4 characters of data_inicial to get the year, store in column Ano_inicial
df_totals['Ano_inicial'] = df_totals['data_inicial'].str[:4]
# Extract the first 4 characters of data_final to get the year, store in column Ano_final
df_totals['Ano_final'] = df_totals['data_final'].str[:4]
print(f"Showing only the first {show_only} rows of {df_totals.shape[0]} total rows")
df_totals[['Ano_inicial', 'Ano_final', 'data_inicial', 'data_final', 'count']].sort_values(by='count', ascending=False).head(show_only)

#### Export to Excel

In [ ]:
df_totals.to_excel("../inferences/jesuita-entrada_totals.xlsx", sheet_name='Sheet_name_1')

## Places of entry, names, nationalities, place of birth, departure

In [ ]:
import pyuca  # to sort accented characters properly
from dehergne_util import get_linked_entity_id
import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)
show_only = 10

jesuita_entrada_full: pd.DataFrame | None = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname'],
    the_type='jesuita-entrada',
    more_attributes=['nacionalidade','nascimento', 'embarque'],
    db=tlnb.db,
)
if jesuita_entrada_full is None:
    raise ValueError("No data found for 'jesuita-entrada' attribute.")
# jesuita_entrada_full.info()


# we use the Collator class to sort the names and countries properly
# since Python does not sort accented characters properly
# we create two new columns with the sort keys
coll = pyuca.Collator()

# first we fill the NaN values with empty strings
jesuita_entrada_full['nacionalidade'] = jesuita_entrada_full['nacionalidade'].fillna('')
jesuita_entrada_full['nascimento'] = jesuita_entrada_full['nascimento'].fillna('')
jesuita_entrada_full['nascimento.comment'] = jesuita_entrada_full['nascimento.comment'].fillna('')
jesuita_entrada_full['name'] = jesuita_entrada_full['name'].fillna('?')

# now we add two columns with sort keys
jesuita_entrada_full['nacionalidade_sort'] = jesuita_entrada_full['nacionalidade'].apply(coll.sort_key)
jesuita_entrada_full['name_sort'] = jesuita_entrada_full['name'].apply(coll.sort_key)


# next we fetch wikidata Ids for place of birth and place of entty
jesuita_entrada_full['jesuita-entrada.wikidata'] = jesuita_entrada_full['jesuita-entrada.comment'].apply(get_linked_entity_id, linked_data_provider='wikidata', if_missing='no wikidata')
# same for the place of birth
jesuita_entrada_full['nascimento.wikidata'] = jesuita_entrada_full['nascimento.comment'].apply(get_linked_entity_id, linked_data_provider='wikidata', if_missing='no wikidata')
# entrada_known = jesuita_entrada_full['jesuita-entrada'] != '?'
only_n = jesuita_entrada_full.fillna('').groupname == 'n'
jesuita_entrada_full = jesuita_entrada_full[only_n]
cols=['jesuita-entrada','jesuita-entrada.wikidata','jesuita-entrada.date','name','nacionalidade','nascimento','nascimento.date','nascimento.wikidata','embarque.date']
# jesuita_entrada_full[(only_n) & (entrada_known)].sort_values(by=['jesuita-entrada','jesuita-entrada.date','name_sort','nascimento'])[cols].head(70)
print(f"Showing only the first {show_only} rows of {jesuita_entrada_full.shape[0]} total rows")
jesuita_entrada_full.sort_values(by=['jesuita-entrada.date','jesuita-entrada','jesuita-entrada','name_sort','nascimento'])[cols].head(show_only)



In [ ]:
jesuita_entrada_full.info()


### Check for place of entry with no wikidata

In [ ]:
# list row with "No wikidata" in the jesuita-entrada-wikidata column or nascimento-wikidata column
if jesuita_entrada_full is None:
    raise ValueError("No data found for 'jesuita-entrada' attribute.")

jesuita_entrada_full[
    (jesuita_entrada_full['jesuita-entrada.wikidata'] == 'no wikidata') &
    (jesuita_entrada_full['jesuita-entrada'] != '?')
][cols].sort_values(by=['jesuita-entrada'])[['jesuita-entrada','name']].head(10)

#### Check for birth places with no wikidata

In [ ]:
# list row with "No wikidata" in the jesuita-entrada-wikidata column or nascimento-wikidata column
no_wikidata_in_birth_place = jesuita_entrada_full[
    (jesuita_entrada_full['nascimento.wikidata'] == 'no wikidata') &
    (jesuita_entrada_full['nascimento'] != '') &
    (jesuita_entrada_full['nascimento'] != '?')
].sort_values(by=['nascimento'])[['nascimento','nascimento.wikidata','name']]
no_wikidata_in_birth_place 

In [ ]:
no_wikidata_in_birth_place.index

## Export to Excel

In [ ]:
from timelink.kleio.utilities import format_timelink_date

if jesuita_entrada_full is not None:
    jesuita_entrada_full.to_excel("../inferences/jesuitas-entrada-full.xlsx")
    # export only those with "nacionalidade" equal to "Portugal"
    jesuita_entrada_full[jesuita_entrada_full['nacionalidade'] == 'Portugal'].to_excel("../inferences/jesuitas-entrada-portugal.xlsx")
    # export only those with "nacionalidade" equal to "Portugal and jesuita-entrada = "?"
    cols = ['name','embarque.date.formated','jesuita-entrada.date.formated', 'jesuita-entrada.date.formated', 'nascimento','nascimento.date.formated','embarque.date','jesuita-entrada', 'jesuita-entrada.date', 'nascimento','nascimento.date']
    # create columns with formated dates embarque.date and jesuita-entrada.date and nascimento.date
    jesuita_entrada_full['nascimento.date.formated'] = jesuita_entrada_full['nascimento.date'].apply(format_timelink_date)
    jesuita_entrada_full['embarque.date.formated'] = jesuita_entrada_full['embarque.date'].apply(format_timelink_date)
    jesuita_entrada_full['jesuita-entrada.date.formated'] = jesuita_entrada_full['jesuita-entrada.date'].apply(format_timelink_date)

    jesuita_entrada_full[
        (jesuita_entrada_full['nacionalidade'] == 'Portugal') &
        (jesuita_entrada_full['jesuita-entrada'] == '?')
    ][cols].sort_values(["embarque.date","jesuita-entrada.date"]).to_excel("../inferences/jesuitas-entrada-portugal-unknown.xlsx")

## Analyse trends in decades

In [ ]:
# extract the year from jesuita-entrada.date
jesuita_entrada_full['ano_entrada'] = jesuita_entrada_full['jesuita-entrada.date'].str[:4]
# convert 'ano_entrada' to integer
jesuita_entrada_full['ano_entrada'] = jesuita_entrada_full['ano_entrada'].astype(int)
# count ano_entrada per periods of 10 years starting from 1540
jesuita_entrada_full['periodo_entrada'] = (jesuita_entrada_full['ano_entrada'] - 1540) // 10 * 10 + 1540
# show columns jesuita-entrada.date, ano_entrada, periodo
jesuita_entrada_full[['jesuita-entrada.date','ano_entrada','periodo_entrada']].sort_values('jesuita-entrada.date').head(10)


In [ ]:
# count number of jesuitas-entrada per period
jesuita_entrada_full.groupby('periodo_entrada')['name'].count().reset_index().sort_values('periodo_entrada')


#### Make a graph ignoring period < 1540


In [ ]:
%pip install seaborn

In [ ]:
# make a chart of the number of jesuitas-entrada per period ignoring period < 1540
# Label the vertical axis as "Número de missionários"
# Label the horizontal axis as "Período de entrada"
import matplotlib.pyplot as plt
import seaborn as sns

# filter period >= 1540
df = jesuita_entrada_full[jesuita_entrada_full.periodo_entrada >= 1540]

# count number of jesuitas-entrada per period
df = df.groupby('periodo_entrada')['name'].count().reset_index().sort_values('periodo_entrada')

# create a bar plot
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x='periodo_entrada', y='name', color='white', edgecolor='black')
plt.xticks(rotation=45)
plt.title('Número de jesuítas por período de entrada')

# Label the vertical axis
plt.ylabel('Número de missionários')

# Label the horizontal axis
plt.xlabel('Período de entrada')

plt.show()


### Place of entry "unkown" with country and place of birth

In [ ]:
import pyuca  # to sort accented characters properly

import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

jesuita_entrada_full = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname'],
    the_type='jesuita-entrada',
    the_value='?',
    more_attributes=['nacionalidade','nascimento','embarque'],
    db=tlnb.db,
)
# filter by those with groupname = 'n' (avoid "referido", "pai", "mãe")
jesuita_entrada_full = jesuita_entrada_full[jesuita_entrada_full.groupname=='n']

jesuita_entrada_full.info()

coll = pyuca.Collator()

# we use the Collator class to sort the names and countries properly
# since Python does not sort accented characters properly
# we create two new columns with the sort keys

# first we fill the NaN values with empty strings
jesuita_entrada_full['nacionalidade'] = jesuita_entrada_full['nacionalidade'].fillna('?')
jesuita_entrada_full['name'] = jesuita_entrada_full['name'].fillna('?')
jesuita_entrada_full['nascimento'] = jesuita_entrada_full['nascimento'].fillna('?')

# now we add two columns with sort keys
jesuita_entrada_full['nacionalidade_sort'] = jesuita_entrada_full['nacionalidade'].apply(coll.sort_key)
jesuita_entrada_full['name_sort'] = jesuita_entrada_full['name'].apply(coll.sort_key)

cols=['name','nacionalidade','nascimento','jesuita-entrada.date','embarque.date','jesuita-entrada.obs']
jesuita_entrada_full[jesuita_entrada_full.groupname=='n'].sort_values(by=['jesuita-entrada.date', 'nacionalidade_sort','name_sort','nascimento'])[cols].head(50)



#### Export to Excel

In [ ]:
jesuita_entrada_full[cols].to_excel("../inferences/jesuitas-entrada-desc.xlsx", sheet_name='Sheet_name_1')

#### Export to text file (Portugal)

In [ ]:
# Get id of interest
country_of_interest = 'Portugal'
ids = jesuita_entrada_full[jesuita_entrada_full.nacionalidade==country_of_interest].index.unique()

tlnb.db.export_as_kleio(ids,f'../inferences/jesuitas-entrada-desc-{country_of_interest}.txt')

##  Missionaries by specific place of entry in the Jesuit order

Obter os missionários por lugar de entrada na ordem


### Example/Exemplo: Coimbra

Showing palce of birth, embarque (departure).

In [ ]:
place_of_entry = "Coimbra"

In [ ]:
import pandas as pd
from timelink.pandas import entities_with_attribute
from dehergne_util import calc_age_at
# show 500 rows
pd.set_option('display.max_rows', 1550)

jesuita_entrada_embarque = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','extra_info'],
    the_type='jesuita-entrada',
    the_value=place_of_entry,
    more_attributes=['dehergne','nascimento','embarque','morte'],
    db=tlnb.db,
)

# jesuita_entrada_embarque.info()

# user only group "n"
jesuita_entrada_embarque = jesuita_entrada_embarque[jesuita_entrada_embarque.groupname=='n']

# add age at 'jesuita-entrada.date'
jesuita_entrada_embarque['age_at_entrada'] = jesuita_entrada_embarque.apply(lambda x: calc_age_at(x['nascimento.date'], x['jesuita-entrada.date']), axis=1)
# convert to int
jesuita_entrada_embarque['age_at_entrada'] = jesuita_entrada_embarque['age_at_entrada'].astype('Int64')

# add age at 'embarque.date'
jesuita_entrada_embarque['age_at_embarque'] = jesuita_entrada_embarque.apply(lambda x: calc_age_at(x['nascimento.date'], x['embarque.date']), axis=1)
# convert to int
jesuita_entrada_embarque['age_at_embarque'] = jesuita_entrada_embarque['age_at_embarque'].astype('Int64')
# years between jesuita-entrada and embarque
jesuita_entrada_embarque['entrada_to_embarque'] = jesuita_entrada_embarque['age_at_embarque'] - jesuita_entrada_embarque['age_at_entrada']

# add age at 'morte.date'
jesuita_entrada_embarque['age_at_morte'] = jesuita_entrada_embarque.apply(lambda x: calc_age_at(x['nascimento.date'], x['morte.date']), axis=1)
# convert to int
jesuita_entrada_embarque['age_at_morte'] = jesuita_entrada_embarque['age_at_morte'].astype('Int64')

# time in missions
jesuita_entrada_embarque['mission_time'] = jesuita_entrada_embarque.apply(lambda x: calc_age_at(x['embarque.date'], x['morte.date']), axis=1)
# convert to int
jesuita_entrada_embarque['mission_time'] = jesuita_entrada_embarque['mission_time'].astype('Int64')


# format dates
jesuita_entrada_embarque['nascimento.date.formated'] = jesuita_entrada_embarque['nascimento.date'].apply(format_timelink_date)
# create column with year (first 4 characters of 'nascimento.date.formated')
jesuita_entrada_embarque['nascimento.date.year'] = jesuita_entrada_embarque['nascimento.date.formated'].str[:4]

jesuita_entrada_embarque['jesuita-entrada.date.formated'] = jesuita_entrada_embarque['jesuita-entrada.date'].apply(format_timelink_date)
# create column with year
jesuita_entrada_embarque['jesuita-entrada.date.year'] = jesuita_entrada_embarque['jesuita-entrada.date.formated'].str[:4]
jesuita_entrada_embarque['embarque.date.formated'] = jesuita_entrada_embarque['embarque.date'].apply(format_timelink_date)
# create column with year
jesuita_entrada_embarque['embarque.date.year'] = jesuita_entrada_embarque['embarque.date.formated'].str[:4]

jesuita_entrada_embarque['morte.date.formated'] = jesuita_entrada_embarque['morte.date'].apply(format_timelink_date)
# create column with year
jesuita_entrada_embarque['morte.date.year'] = jesuita_entrada_embarque['morte.date.formated'].str[:4]
# add column with wikidata id for place of entry
jesuita_entrada_embarque['jesuita-entrada.wikidata'] = jesuita_entrada_embarque['jesuita-entrada.comment'].apply(get_linked_entity_id, linked_data_provider='wikidata', if_missing=None)
# add column for wikidata of place of birth
jesuita_entrada_embarque['nascimento.wikidata'] = jesuita_entrada_embarque['nascimento.comment'].apply(get_linked_entity_id, linked_data_provider='wikidata', if_missing=None)

cols=['dehergne','name','nascimento.date.year','nascimento', 'nascimento.wikidata',
            'jesuita-entrada.date.year','jesuita-entrada','jesuita-entrada.wikidata',
            'age_at_entrada','entrada_to_embarque',
            'embarque.date.year','age_at_embarque',
            'morte.date.year','age_at_morte','mission_time','morte']
# remove lines with duplicate values for column 'dehergne'
jesuita_entrada_embarque = jesuita_entrada_embarque.sort_values(by=['jesuita-entrada.date.formated','embarque.date.formated']).drop_duplicates(subset=['dehergne'])
jesuita_entrada_embarque.info()
jesuita_entrada_embarque.sort_values(by=['name','jesuita-entrada.date.formated','embarque.date.formated'])[cols]


In [ ]:
# print a list of name and jesuita-entrada.date.year
place_of_entry_ids = jesuita_entrada_embarque.index.unique()
print("Number of people in the list:", len(place_of_entry_ids))
# number print rows
i = 0
for id, name,year,embarque in jesuita_entrada_embarque[['name','jesuita-entrada.date.year','embarque.date.year']].sort_values('name').itertuples(index=True):
    # print number of interactions
    i += 1
    print(f"{i:2} {name} {year}-{embarque} {id}")

### Analyse trends in decades for specific place of entry

In [ ]:
# Extract year from jesuita-entrada.date
jesuita_entrada_embarque['ano_entrada'] = jesuita_entrada_embarque['jesuita-entrada.date'].str[:4]
# Convert 'ano_entrada' to integer
jesuita_entrada_embarque['ano_entrada'] = jesuita_entrada_embarque['ano_entrada'].astype(int)
# Count 'ano_entrada' per periods of 10 years starting from 1540
jesuita_entrada_embarque['periodo_entrada'] = (jesuita_entrada_embarque['ano_entrada'] - 1540) // 10 * 10 + 1540
# Show columns jesuita-entrada.date, ano_entrada, periodo
jesuita_entrada_embarque[['jesuita-entrada.date','ano_entrada','periodo_entrada']].sort_values('jesuita-entrada.date').head(10)


In [ ]:
# count number of jesuitas-entrada per period
jesuita_entrada_embarque.groupby('periodo_entrada')['name'].count().reset_index().sort_values('periodo_entrada')

In [ ]:
# fill missing periods with 0
df_period_place_of_entry = jesuita_entrada_embarque.groupby('periodo_entrada')['name'].count().reset_index().sort_values('periodo_entrada')
df_period_place_of_entry = df_period_place_of_entry.set_index('periodo_entrada').reindex(range(1540, 1790, 10)).fillna(0).reset_index()
df_period_place_of_entry

#### Make a graph

In [ ]:
# make a chart of the number of jesuitas-entrada per period ignoring period < 1540
# Label the vertical axis as "Número de missionários de Coimbra"
# Label the horizontal axis as "Período de entrada"
import matplotlib.pyplot as plt
import seaborn as sns

# create a bar plot
plt.figure(figsize=(12, 6))
sns.barplot(data=df_period_place_of_entry, x='periodo_entrada', y='name', color='white', edgecolor='black')
plt.xticks(rotation=45)
plt.title(f'Número de jesuítas de {place_of_entry} por período de entrada')

# Label the vertical axis
plt.ylabel(f'Número de missionários de {place_of_entry}')

# Label the horizontal axis
plt.xlabel('Período de entrada')

plt.show()

### Export Place of Entry information

In [ ]:
jesuita_entrada_embarque[cols].to_excel(f"../inferences/jesuita-entrada-{place_of_entry}.xlsx", sheet_name=f'jesuita_entrada_{place_of_entry}', index=True)

## Create a map with place of entry and place of birth.

Use wikidata id to fetch coordinates of places from wikidata

### Export to markdown


In [ ]:
# pip install tabulate
%pip install tabulate


In [ ]:
import os
from pandas import DataFrame
from timelink.pandas import group_attributes as person_attributes

# create if it does not exist directory ../inferences/{place_of_entry}/markdown"
directory = f"../inferences/{place_of_entry}/markdown"
if not os.path.exists(directory):
    os.makedirs(directory)


for id, name,year,embarque in jesuita_entrada_embarque[['name','jesuita-entrada.date.year','embarque.date.year']].sort_values('name').itertuples(index=True):
    with tlnb.db.session() as session:
        p = tlnb.db.get_person(id, session=session)
        session.add(p)
        pk:str = p.to_kleio()
        pk = f"""## {name}
^[In Edit mode, select title (including the ##), right click and "Extract selection"]

---
### {name} @ timelink
```
{pk}
```

---
Select title, right click on selected title, choose extract note
"""
        fname = f"{directory}/{id}.md"
        with open(fname, 'w', encoding='utf-8') as f:
            f.write(pk)



In [ ]:
jesuita_entrada_embarque.info()

In [ ]:
# export to markdown table dataframe jesuita_entrada_embarque
def df_to_markdown_table(df, cols=None):
    if cols is not None:
        df = df[cols]
    return df.to_markdown(index=True, tablefmt="pipe")
jesuita_entrada_embarque['name_link'] = jesuita_entrada_embarque['name'].apply(lambda x: f"[[{x}]]")  # obsidian style link
# update columns to include the link
cols = ['name_link', 'jesuita-entrada.date.formated', 'embarque.date.formated', 'nascimento.date.formated', 'age_at_entrada', 'age_at_embarque', 'age_at_morte', 'mission_time', 'dehergne']
markdown_table = df_to_markdown_table(jesuita_entrada_embarque, cols=cols)
# save markdown table to file
with open(f"{directory}/00_{place_of_entry}_list.md", "w") as f:
    f.write(markdown_table)

## Export people to text file as Kleio

In [ ]:
tlnb.db.export_as_kleio(jesuita_entrada_embarque.index.unique(), f"../inferences/jesuita-entrada-{place_of_entry}.txt")

## Export those with place of entry originally unkown by Dehergne

Record in which the place of entry was added based on extra sources of information.

In [ ]:
# List rows if '"original": "?"' is in column 'jesuita_entrada.obs'
# jesuita_entrada_comment = 'jesuita-entrada.original.comment'  # bug in timelink-py should be 'jesuita-entrada.comment'
jesuita_entrada_comment = 'jesuita-entrada.comment'  # use when bug is fixed

cols_2 = ['name','jesuita-entrada','jesuita-entrada.date','jesuita-entrada.original',jesuita_entrada_comment, 'jesuita-entrada.extra_info']
place_of_entry_extra = jesuita_entrada_embarque[jesuita_entrada_embarque['jesuita-entrada.original'] == "?"][cols_2]
place_of_entry_extra_ids = place_of_entry_extra.index.unique()
print("jesuita-entrada through extra info:",len(place_of_entry_extra_ids))
place_of_entry_extra

#### Save to Excel

In [ ]:
place_of_entry_extra.to_excel(f"../inferences/jesuita-entrada-{place_of_entry}-extra_info.xlsx", sheet_name=f'jesuita_entrada_{place_of_entry}_extra', index=True)

Make a list

In [ ]:
names_list = sorted(list(place_of_entry_extra.name))

# Join the list into a comma-separated string
comma_separated_names = ', '.join(sorted(names_list))

print(comma_separated_names)
names_list

#### export to text file

In [ ]:

tlnb.db.export_as_kleio(place_of_entry_extra_ids, f"../inferences/jesuita-entrada-{place_of_entry}-extra_info.txt")

#### If place of entry unkown check place of entry in the birthplace 

Check for place of interest in the birth place 
of those with no record of place of entry.

Those born in the same city, district or diocesis as the place 
of entry under analysis are good candidates for further research.


In [ ]:
# filter dataframe jesuita_entrada_desc for rows with place_of_entry in in column 'nascimento'
maybe = jesuita_entrada_full[jesuita_entrada_full['nascimento'].str.contains(place_of_entry, na=False)].index.unique()
jesuita_entrada_comment = 'jesuita-entrada.comment'  # use when bug is fixed

cols=['name','nacionalidade','nascimento','jesuita-entrada.date','jesuita-entrada','jesuita-entrada.original',jesuita_entrada_comment,'jesuita-entrada.obs']
maybe_place_of_entry = jesuita_entrada_full.loc[maybe]
maybe_place_of_entry.info()
maybe_place_of_entry.sort_values(by=['jesuita-entrada.date'])[cols]

In [ ]:
print("Number of unique ids:", maybe_place_of_entry.index.nunique())

Export maybe place of entry candidates

In [ ]:
maybe_place_of_entry[cols].to_excel(f"../inferences/jesuita-entrada-maybe-{place_of_entry}.xlsx", sheet_name=f'jesuita_entrada_maybe_{place_of_entry}', index=True)

Export as text file

In [ ]:
tlnb.db.export_as_kleio(maybe_place_of_entry.index.unique(), f"../inferences/jesuita-entrada-maybe-{place_of_entry}.txt")

 ## Roles and tasks of missionaries from specific place of entry

In [ ]:
import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 1550)

ids = place_of_entry_ids

roles_tasks = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname',],
    the_type=['jesuita-cargo','jesuita-tarefa'],
    column_name='cargos_tarefas',
    more_attributes=[],
    filter_by=ids,
    db=tlnb.db,
)
# remove rows with "cargos_tarefas" NaN
roles_tasks = roles_tasks.dropna(subset=['cargos_tarefas'])
# extract the year from cargos_tarefas.date
roles_tasks['cargos_tarefas.date.year'] = roles_tasks['cargos_tarefas.date'].str[:4]

roles_tasks.info()
cols=['name','cargos_tarefas','cargos_tarefas.type','cargos_tarefas.date.year','groupname',]
# show those without date
# roles_tasks.loc[roles_tasks['cargos_tarefas.date'] == "0", cols].sort_values(['cargos_tarefas.type','cargos_tarefas','cargos_tarefas.date','name']).head(150)
roles_tasks[cols].sort_values(['cargos_tarefas.type','cargos_tarefas','cargos_tarefas.date.year']).head(150)

#### Export to Excel

In [ ]:
roles_tasks[cols].to_excel(f"../inferences/jesuita-entrada-{place_of_entry}-cargos_tarefas.xlsx", sheet_name='cargos_tarefas', index=True)

#### Get start and end dates for roles

In [ ]:
roles_start_end = roles_tasks[roles_tasks['cargos_tarefas.type']=='jesuita-cargo'].groupby(['name','cargos_tarefas']).agg(
    start_date=('cargos_tarefas.date.year', 'min'),
    end_date=('cargos_tarefas.date.year', 'max')
).reset_index().sort_values(['name','cargos_tarefas','start_date',])
roles_start_end

##### Save to Excel

In [ ]:
roles_start_end.to_excel(f"../inferences/jesuita-entrada-{place_of_entry}-cargos_tarefas.datas.xlsx", sheet_name='cargos_tarefas', index=True)

#### Filter by role

In [ ]:
# filter
regex = "(.*rovincial|.*Visitador|.*Superior|.*Procurador).*(China.*|Japão.*)"
roles_filtered = roles_start_end[roles_start_end.cargos_tarefas.str.contains(regex, na=False)]
roles_filtered.sort_values(['cargos_tarefas', 'start_date'])

## Joint biography of entry in Place of interest

In [ ]:
from timelink.pandas import display_group_attributes, group_attributes

ids = place_of_entry_ids

no_show=['jesuita-entrada.level','jesuita-entrada.line','dehergne','estadia@wikidata','jesuita-estatuto',
          'nome','nacionalidade','nascimento@wikidata','nome-chines','titulo','jesuita-entrada@wikidata',
          'jesuita-votos-local@wikidata',
          'wicky','wicky-viagem']

show=['jesuita-entrada','embarque','partida','chegada','estadia','cargo','jesuita-cargo','jesuita-tarefa','morte']
display_group_attributes(ids,
                    entity_type='person',
                    header_elements=['name'],
                    sort_header='jesuita-entrada.date',
                    header_attributes=['jesuita-entrada'],
                    include_attributes=show,
                    exclude_attributes=no_show,
                    sort_attributes=['the_date','the_type','the_value'],
                    db=tlnb.db,
                    category='id',
                    cmap_name='Pastel2'
                    )